# What Survives the Guardrail Shift?

## A post-competition transfer study for AI Agent Security

A public leaderboard rewards performance on one visible security surface. A hidden evaluation can reward a very different part of the solution space. This notebook studies that shift as a **portfolio-selection problem**: how did two fixed candidate portfolios behave across eight heterogeneous guardrail surfaces?

Data: [AI Agent Security: Guardrail Transfer Study](https://www.kaggle.com/datasets/syouyatobita/ai-agent-security-guardrail-transfer-study), released under CC BY 4.0.

> **Responsible-use scope.** This is an offline analysis of the Kaggle benchmark. It contains no raw prompts, payloads, tool messages, credentials, Competition Data, or instructions for attacking real systems. Six policy surfaces are author-created synthetic proxies; they are not recovered or verified hidden Private guardrails.

## Competition result and study context

The author received a **Bronze medal**, with a final selected Private score of **6.480**. The best Public score across submissions was **86.085**, from a different submission. The official selected-submission records checked on **2026-09-07** were:

| Submission ref | Public score | Private score |
|---|---:|---:|
| 55811977 | 37.400 | 6.480 |
| 55889746 | 44.995 | 0.000 |

Public 86.085 belongs to ref 55783911. It is a team-level best across submissions; it must not be paired with Private 6.480 as a same-submission transfer measurement.

In the **2026-09-07** [official Private leaderboard](https://www.kaggle.com/competitions/ai-agent-security-multi-step-tool-attacks/leaderboard?tab=private) API snapshot, the author's team was at **position 349 among 4,187 returned entries**. This position was computed from the API's returned order. The API has no explicit rank field; its entry count and position have not been reconciled with the UI display.

The E0348 experiment below compares a local baseline with a local mixed portfolio. It is separate from the medal-producing submission and does not explain the final rank causally. Its `private_guardrail_observed` field is always `false`: the hidden guardrail implementation was not observed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

candidate_dirs = [
    Path("/kaggle/input/datasets/syouyatobita/ai-agent-security-guardrail-transfer-study"),
    Path("/kaggle/input/ai-agent-security-guardrail-transfer-study"),
    Path("../dataset"),
    Path("publication/dataset"),
]
DATA_DIR = next((path for path in candidate_dirs if (path / "transfer_summary.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Attach the guardrail-transfer Dataset or run from this publication package.")

runs = pd.read_csv(DATA_DIR / "transfer_runs.csv")
summary = pd.read_csv(DATA_DIR / "transfer_summary.csv")
catalog = pd.read_csv(DATA_DIR / "guardrail_surface_catalog.csv")

print(f"Loaded {len(runs)} runs and {len(summary)} surface comparisons from {DATA_DIR}")

## 1. Integrity checks before interpretation

The study used exactly 2,000 generated candidates per fixed variant and a 300-second prefix per surface. `Optimal` has two baseline bookends; every other variant/surface pair has one run. These checks make silent partial-data errors visible.

In [ ]:
expected_surfaces = {
    "Optimal", "Rules", "map_semantic", "dlp_no_intent",
    "pure_provenance", "intent_control", "rules_authz", "sanitize_taint",
}

assert len(runs) == 17
assert len(summary) == 8
assert set(summary["surface"]) == expected_surfaces
assert runs["generated_candidates"].eq(2_000).all()
assert runs["prefix_budget_seconds"].eq(300).all()
assert runs["all_paths_pass"].all()
assert runs["positive_cells_unique"].all()
assert (~runs["private_guardrail_observed"]).all()
assert (~summary["private_guardrail_observed"]).all()
assert runs["row_digest"].is_unique

optimal_bookends = runs.query(
    "surface == 'Optimal' and variant == 'single_family_baseline'"
)["normalized_proxy_score"]
bookend_ratio = optimal_bookends.iloc[-1] / optimal_bookends.iloc[0]

pd.Series({
    "run_rows": len(runs),
    "surface_comparisons": len(summary),
    "valid_path_rate": runs["all_paths_pass"].mean(),
    "unique_positive_cell_rate": runs["positive_cells_unique"].mean(),
    "Optimal_end_to_start_bookend_ratio": bookend_ratio,
}, name="validation").to_frame()

## 2. Transfer under the frozen hypothesis weights

The six synthetic worlds were frozen with weights summing to one. Under that experimental distribution, the mixed portfolio improves the weighted proxy score. This is a fixed-weight comparison under stated assumptions. Alternative weights were not evaluated here, and the weights do not estimate the true hidden policy distribution.

In [ ]:
synthetic = summary.dropna(subset=["hypothesis_weight"]).copy()
assert np.isclose(synthetic["hypothesis_weight"].sum(), 1.0)

baseline_expected = np.average(
    synthetic["single_family_baseline_score"],
    weights=synthetic["hypothesis_weight"],
)
portfolio_expected = np.average(
    synthetic["static_mixed_portfolio_score"],
    weights=synthetic["hypothesis_weight"],
)
weighted_ratio = portfolio_expected / baseline_expected

weighted_result = pd.DataFrame({
    "metric": ["weighted synthetic proxy score"],
    "single-family baseline": [baseline_expected],
    "static mixed portfolio": [portfolio_expected],
    "portfolio / baseline": [weighted_ratio],
    "relative change": [weighted_ratio - 1.0],
})
weighted_result.style.format({
    "single-family baseline": "{:.3f}",
    "static mixed portfolio": "{:.3f}",
    "portfolio / baseline": "{:.3f}×",
    "relative change": "{:+.1%}",
})

In [ ]:
plot_data = summary.sort_values("absolute_delta").set_index("surface")
colors = ["#7587a3", "#d97735"]
ax = plot_data[[
    "single_family_baseline_score",
    "static_mixed_portfolio_score",
]].plot.barh(figsize=(10, 6), color=colors, width=0.78)
ax.set_title("A fixed portfolio trades performance across guardrail surfaces", pad=14)
ax.set_xlabel("Normalized proxy score")
ax.set_ylabel("Guardrail surface")
ax.legend(["Single-family baseline", "Static mixed portfolio"], loc="lower right")
for side in ("top", "right"): ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

Across the eight surfaces, the portfolio scores higher on four, lower on three, and ties on one. Three details help explain the aggregate:

1. The portfolio improves `Optimal` and `pure_provenance` by about 9%, while producing positive findings where the baseline scored zero on two stricter synthetic proxies.
2. It gives back roughly 12% on `Rules`, `rules_authz`, and `sanitize_taint`. Diversity has an opportunity cost.
3. Both variants score zero on `intent_control`. Adding families does not help when all families share the same blind spot.

In [ ]:
diagnostic = summary[[
    "surface", "surface_type", "hypothesis_weight",
    "single_family_baseline_score", "static_mixed_portfolio_score", "absolute_delta",
]].copy()
diagnostic["relative_change"] = np.where(
    diagnostic["single_family_baseline_score"] > 0,
    diagnostic["absolute_delta"] / diagnostic["single_family_baseline_score"],
    np.nan,
)
diagnostic["outcome"] = np.select(
    [
        diagnostic["absolute_delta"] > 0,
        diagnostic["absolute_delta"] < 0,
    ],
    ["portfolio higher", "baseline higher"],
    default="tie",
)
diagnostic.sort_values("absolute_delta", ascending=False).style.format({
    "hypothesis_weight": "{:.0%}",
    "single_family_baseline_score": "{:.2f}",
    "static_mixed_portfolio_score": "{:.2f}",
    "absolute_delta": "{:+.2f}",
    "relative_change": "{:+.1%}",
})

## 3. Throughput is part of the result

The evaluation was time bounded, so a portfolio can alter both the quality mix and the number of entries completed. The table below keeps those effects visible instead of treating every score difference as a pure policy effect.

In [ ]:
throughput = (
    runs.assign(entries_per_second=runs["completed_entries"] / runs["elapsed_seconds"])
    .groupby(["surface", "variant"], as_index=False)
    .agg(
        mean_completed_entries=("completed_entries", "mean"),
        mean_entries_per_second=("entries_per_second", "mean"),
        mean_normalized_proxy_score=("normalized_proxy_score", "mean"),
    )
)
throughput.pivot(index="surface", columns="variant").round(3)

## 4. What I would carry into the next competition

- Treat public score as one sample from a family of evaluator policies, not as a sufficient statistic.
- Freeze a small set of meaningfully different local guardrail proxies before comparing candidates.
- Measure a portfolio's floor and shared blind spots, not only its weighted mean.
- Keep throughput, structural validity, and uniqueness gates alongside score.
- Separate a preregistered confirmatory comparison from adaptive exploration.

The key lesson is not that this particular 2/7–5/7 mix is optimal. It clearly is not universal. The lesson is that **guardrail diversity changes which candidate families retain value**, and this can be measured without pretending to know the hidden evaluator.

## Limitations

- Six of eight surfaces are synthetic hypotheses with subjective frozen weights.
- This completion experiment used Gemma only, seed 123, and 300-second prefixes.
- Scores combine candidate behavior, guardrail decisions, and time-bounded throughput.
- No confidence interval is claimed from a single run per ordinary surface. The Optimal bookend checks drift but does not replace replication.
- The hidden Private guardrail was never observed. The final leaderboard result cannot identify its implementation.
- Raw payloads are intentionally withheld for rule compliance and responsible communication, so this release reproduces the analysis rather than the full candidate-generation process.

Dataset: **AI Agent Security: Guardrail Transfer Study**, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Notebook code: [MIT](https://opensource.org/license/mit), copyright (c) 2026 Shoya Tobita.